# MCQ Big-Run Visualizer (Recent 1000 Points)

This notebook visualizes MCQ-construction results for recent **big runs** (`mcq_sweep`) and inspects the most recent ~1000 answer-points per run.

What it does:
- Lists recent big MCQ sweeps and run counts (so you can validate "should be 140").
- Loads `mcq_run` payloads linked to a selected sweep.
- Builds run-level QC metrics and label-distribution views.
- Lets you drill into one run's per-sample label behavior.

Notes:
- Uses `Group` / `GroupLink` records in the v2 schema.
- Uses the run payload's `probe_details.summary.per_sample_label_counts` when available.

In [2]:
from __future__ import annotations

import math
import os
import sys
from pathlib import Path
from typing import Any

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from dotenv import load_dotenv
from sqlalchemy import create_engine, func, select
from sqlalchemy.orm import Session

pd.set_option("display.max_colwidth", 120)


def find_repo_root(start: Path) -> Path:
    for path in [start, *start.parents]:
        if (path / "src" / "study_query_llm").exists():
            return path
    raise RuntimeError("Could not find repo root containing src/study_query_llm")


REPO_ROOT = find_repo_root(Path.cwd().resolve())
SRC_PATH = REPO_ROOT / "src"
if str(SRC_PATH) not in sys.path:
    sys.path.insert(0, str(SRC_PATH))

load_dotenv(REPO_ROOT / ".env")
DATABASE_URL = os.getenv("DATABASE_URL")
if not DATABASE_URL:
    raise RuntimeError("DATABASE_URL is missing. Set it in .env or your environment.")

from study_query_llm.db.models_v2 import Group, GroupLink  # noqa: E402

engine = create_engine(DATABASE_URL)

print(f"Repo root: {REPO_ROOT}")
print(f"Using DATABASE_URL prefix: {DATABASE_URL.split(':', 1)[0]}")


Repo root: C:\Users\spenc\Cursor Repos\study-query-llm
Using DATABASE_URL prefix: postgresql


In [3]:
def _safe_meta(value: Any) -> dict[str, Any]:
    return dict(value) if isinstance(value, dict) else {}


def _summary_from_meta(meta: dict[str, Any]) -> dict[str, Any]:
    probe_details = meta.get("probe_details")
    if isinstance(probe_details, dict):
        summary = probe_details.get("summary")
        if isinstance(summary, dict):
            return dict(summary)
    summary = meta.get("result_summary")
    return dict(summary) if isinstance(summary, dict) else {}


def _to_int(value: Any, default: int = 0) -> int:
    try:
        return int(value)
    except Exception:
        return default


def _to_float(value: Any, default: float = math.nan) -> float:
    try:
        return float(value)
    except Exception:
        return default


def list_recent_mcq_big_runs(limit: int = 10, min_runs: int = 100) -> pd.DataFrame:
    with Session(engine) as session:
        sweep_filters = [Group.group_type == "mcq_sweep"]
        deleted_at_col = getattr(Group, "deleted_at", None)
        if deleted_at_col is not None:
            sweep_filters.append(deleted_at_col.is_(None))

        sweeps = (
            session.execute(
                select(Group)
                .where(*sweep_filters)
                .order_by(Group.created_at.desc())
            )
            .scalars()
            .all()
        )
        if not sweeps:
            return pd.DataFrame(
                columns=[
                    "sweep_id",
                    "sweep_name",
                    "created_at",
                    "contains_runs",
                    "is_about_140",
                ]
            )

        sweep_ids = [int(s.id) for s in sweeps]
        link_rows = session.execute(
            select(GroupLink.parent_group_id, func.count(GroupLink.id))
            .where(
                GroupLink.parent_group_id.in_(sweep_ids),
                GroupLink.link_type == "contains",
            )
            .group_by(GroupLink.parent_group_id)
        ).all()
        run_count_by_sweep = {int(parent_id): int(n) for parent_id, n in link_rows}

    rows: list[dict[str, Any]] = []
    for sweep in sweeps[:limit]:
        run_count = int(run_count_by_sweep.get(int(sweep.id), 0))
        if run_count < min_runs:
            continue
        rows.append(
            {
                "sweep_id": int(sweep.id),
                "sweep_name": str(sweep.name),
                "created_at": sweep.created_at,
                "contains_runs": run_count,
                "is_about_140": abs(run_count - 140) <= 2,
            }
        )

    if not rows:
        return pd.DataFrame(
            columns=[
                "sweep_id",
                "sweep_name",
                "created_at",
                "contains_runs",
                "is_about_140",
            ]
        )
    return pd.DataFrame(rows).sort_values("created_at", ascending=False).reset_index(drop=True)


def build_recent_window_frames(
    sweep_id: int,
    recent_points_per_run: int = 1000,
) -> tuple[pd.DataFrame, pd.DataFrame, pd.DataFrame]:
    with Session(engine) as session:
        run_filters = [
            GroupLink.parent_group_id == int(sweep_id),
            GroupLink.link_type == "contains",
            Group.group_type == "mcq_run",
        ]
        deleted_at_col = getattr(Group, "deleted_at", None)
        if deleted_at_col is not None:
            run_filters.append(deleted_at_col.is_(None))

        runs = (
            session.execute(
                select(Group)
                .join(GroupLink, GroupLink.child_group_id == Group.id)
                .where(*run_filters)
                .order_by(Group.id.asc())
            )
            .scalars()
            .all()
        )

    run_rows: list[dict[str, Any]] = []
    label_rows: list[dict[str, Any]] = []
    sample_rows: list[dict[str, Any]] = []

    for run in runs:
        meta = _safe_meta(run.metadata_json)
        summary = _summary_from_meta(meta)

        run_key = str(meta.get("run_key") or run.name or f"run_{run.id}")
        deployment = str(meta.get("deployment") or summary.get("deployment") or "")
        subject = str(meta.get("subject") or summary.get("subject") or "")
        level = str(meta.get("level") or summary.get("level") or "")

        labels = [str(x).upper() for x in (summary.get("labels") or []) if str(x).strip()]
        per_sample_raw = summary.get("per_sample_label_counts") or []
        per_sample = [row for row in per_sample_raw if isinstance(row, dict)]

        if not labels and per_sample:
            labels = sorted({str(k).upper() for row in per_sample for k in row.keys()})
        if not labels:
            n_options = _to_int(meta.get("options_per_question"), 5)
            labels = [chr(ord("A") + i) for i in range(max(1, n_options))]

        question_count = _to_int(summary.get("question_count"), _to_int(meta.get("questions_per_test"), 0))
        if question_count <= 0 and per_sample:
            question_count = _to_int(sum(_to_int(v, 0) for v in per_sample[0].values()), 0)

        samples_requested = _to_int(
            summary.get("samples_requested"),
            _to_int(meta.get("samples_per_combo"), len(per_sample)),
        )
        samples_valid = _to_int(summary.get("samples_with_valid_answer_key"), len(per_sample))
        expected_points = question_count * samples_requested if question_count > 0 else 0
        realized_points = _to_int(summary.get("answer_count_total"), 0)
        if realized_points <= 0 and question_count > 0:
            realized_points = question_count * samples_valid

        if question_count > 0:
            samples_for_window = int(math.ceil(float(recent_points_per_run) / float(question_count)))
        else:
            samples_for_window = len(per_sample)
        samples_for_window = min(len(per_sample), max(1, samples_for_window)) if per_sample else 0
        window_samples = per_sample[-samples_for_window:] if samples_for_window else []
        first_sample_idx = max(1, len(per_sample) - len(window_samples) + 1)

        window_counts: dict[str, int] = {label: 0 for label in labels}
        window_points = 0
        for offset, sample_counts in enumerate(window_samples):
            all_labels = list(labels)
            for raw_label in sample_counts.keys():
                label = str(raw_label).upper()
                if label not in all_labels:
                    all_labels.append(label)
                    window_counts.setdefault(label, 0)

            sample_idx = first_sample_idx + offset
            sample_total = 0
            for label in all_labels:
                count = _to_int(sample_counts.get(label), 0)
                sample_total += count
                window_counts[label] = _to_int(window_counts.get(label), 0) + count
                sample_rows.append(
                    {
                        "sweep_id": int(sweep_id),
                        "run_id": int(run.id),
                        "run_key": run_key,
                        "deployment": deployment,
                        "subject": subject,
                        "level": level,
                        "sample_idx": int(sample_idx),
                        "label": label,
                        "count": int(count),
                    }
                )
            window_points += sample_total

        target_window_points = min(recent_points_per_run, expected_points) if expected_points > 0 else recent_points_per_run
        for label in sorted(window_counts.keys()):
            count = int(window_counts[label])
            label_rows.append(
                {
                    "sweep_id": int(sweep_id),
                    "run_id": int(run.id),
                    "run_key": run_key,
                    "label": label,
                    "window_count": count,
                    "window_points": int(window_points),
                    "window_pct": (float(count) / float(window_points)) if window_points > 0 else math.nan,
                    "target_window_points": int(target_window_points),
                }
            )

        run_rows.append(
            {
                "sweep_id": int(sweep_id),
                "run_id": int(run.id),
                "run_key": run_key,
                "deployment": deployment,
                "subject": subject,
                "level": level,
                "options_per_question": _to_int(meta.get("options_per_question"), 0),
                "questions_per_test": question_count,
                "samples_requested": samples_requested,
                "samples_valid": samples_valid,
                "expected_answer_points": int(expected_points),
                "realized_answer_points": int(realized_points),
                "recent_window_target_points": int(target_window_points),
                "recent_window_points": int(window_points),
                "sample_rows_available": int(len(per_sample)),
                "samples_in_window": int(len(window_samples)),
                "parse_failure_count": _to_int(summary.get("parse_failure_count"), 0),
                "call_error_count": _to_int(summary.get("call_error_count"), 0),
                "chi_square_vs_uniform": _to_float(summary.get("chi_square_vs_uniform")),
            }
        )

    run_df = pd.DataFrame(run_rows).sort_values("run_key").reset_index(drop=True) if run_rows else pd.DataFrame()
    label_window_df = (
        pd.DataFrame(label_rows).sort_values(["run_key", "label"]).reset_index(drop=True)
        if label_rows
        else pd.DataFrame()
    )
    sample_long_df = (
        pd.DataFrame(sample_rows).sort_values(["run_key", "sample_idx", "label"]).reset_index(drop=True)
        if sample_rows
        else pd.DataFrame()
    )
    return run_df, label_window_df, sample_long_df


def plot_single_run_window(run_key: str, sample_long_df: pd.DataFrame, rolling_samples: int = 5) -> None:
    sub = sample_long_df[sample_long_df["run_key"] == run_key].copy()
    if sub.empty:
        print(f"No per-sample rows for run_key={run_key}")
        return

    pivot = sub.pivot_table(index="sample_idx", columns="label", values="count", aggfunc="sum").fillna(0.0)
    denom = pivot.sum(axis=1).replace(0.0, np.nan)
    pct = pivot.div(denom, axis=0) * 100.0
    smooth = pct.rolling(rolling_samples, min_periods=1).mean()

    fig, axes = plt.subplots(1, 2, figsize=(15, 4))

    for label in pct.columns:
        axes[0].plot(
            pct.index,
            pct[label],
            marker="o",
            markersize=3,
            linewidth=1,
            alpha=0.5,
            label=label,
        )
    axes[0].set_title(f"{run_key} - label % per sample")
    axes[0].set_xlabel("Sample index")
    axes[0].set_ylabel("Percent")
    axes[0].set_ylim(0, 100)
    axes[0].grid(alpha=0.2)
    axes[0].legend(ncol=min(5, len(pct.columns)), fontsize=8)

    stack_values = [smooth[c].fillna(0.0).to_numpy() for c in smooth.columns]
    axes[1].stackplot(smooth.index.to_numpy(), stack_values, labels=smooth.columns, alpha=0.9)
    axes[1].set_title(f"{run_key} - rolling blend ({rolling_samples} samples)")
    axes[1].set_xlabel("Sample index")
    axes[1].set_ylabel("Percent")
    axes[1].set_ylim(0, 100)
    axes[1].legend(loc="upper left", ncol=min(5, len(smooth.columns)), fontsize=8)

    plt.tight_layout()
    plt.show()


In [4]:
BIG_RUN_MIN_RUNS = 100
RECENT_SWEEP_LIMIT = 10

big_runs_df = list_recent_mcq_big_runs(limit=RECENT_SWEEP_LIMIT, min_runs=BIG_RUN_MIN_RUNS)
display(big_runs_df)

if big_runs_df.empty:
    raise RuntimeError("No mcq_sweep groups found with >=100 runs in this database.")


AttributeError: type object 'Group' has no attribute 'deleted_at'

In [ ]:
# Pick a sweep_id here (defaults to the most recent big run)
SWEEP_ID = int(big_runs_df.iloc[0]["sweep_id"])
RECENT_POINTS_PER_RUN = 1000

run_df, label_window_df, sample_long_df = build_recent_window_frames(
    sweep_id=SWEEP_ID,
    recent_points_per_run=RECENT_POINTS_PER_RUN,
)

print(f"Selected sweep_id={SWEEP_ID}")
print(f"Runs in sweep: {len(run_df)} (target is often about 140)")
print(f"Per-run point window target: {RECENT_POINTS_PER_RUN}")
display(run_df.head(10))


In [ ]:
if run_df.empty:
    print("No runs found for this sweep.")
else:
    fig, axes = plt.subplots(2, 2, figsize=(18, 10))

    realized = run_df["realized_answer_points"].fillna(0).astype(float)
    axes[0, 0].hist(realized, bins=20, color="#4C72B0", alpha=0.85)
    axes[0, 0].axvline(RECENT_POINTS_PER_RUN, color="red", linestyle="--", label="1000-point target")
    axes[0, 0].set_title("Realized answer points per run")
    axes[0, 0].set_xlabel("answer_count_total")
    axes[0, 0].set_ylabel("Runs")
    axes[0, 0].legend()

    axes[0, 1].scatter(np.arange(len(run_df)), realized, s=28, alpha=0.8)
    axes[0, 1].axhline(RECENT_POINTS_PER_RUN, color="red", linestyle="--")
    axes[0, 1].set_title("Run index vs realized answer points")
    axes[0, 1].set_xlabel("Run index")
    axes[0, 1].set_ylabel("answer_count_total")
    axes[0, 1].grid(alpha=0.2)

    by_subject = run_df.groupby("subject", dropna=False).size().sort_values(ascending=False)
    axes[1, 0].bar(by_subject.index.astype(str), by_subject.values, color="#55A868", alpha=0.9)
    axes[1, 0].set_title("Runs per subject in selected big run")
    axes[1, 0].set_xlabel("Subject")
    axes[1, 0].set_ylabel("Runs")
    axes[1, 0].tick_params(axis="x", rotation=45)

    pivot = (
        label_window_df.pivot(index="run_key", columns="label", values="window_pct")
        .fillna(0.0)
        .sort_index()
        * 100.0
    )
    if pivot.empty:
        axes[1, 1].text(0.5, 0.5, "No label-window data", ha="center", va="center")
        axes[1, 1].set_axis_off()
    else:
        im = axes[1, 1].imshow(pivot.values, aspect="auto", cmap="viridis", vmin=0, vmax=100)
        axes[1, 1].set_title(f"Label % by run (recent ~{RECENT_POINTS_PER_RUN} points)")
        axes[1, 1].set_xlabel("Label")
        axes[1, 1].set_ylabel("Run (sorted by run_key)")
        axes[1, 1].set_xticks(np.arange(len(pivot.columns)))
        axes[1, 1].set_xticklabels(pivot.columns.tolist())
        axes[1, 1].set_yticks([])
        cbar = fig.colorbar(im, ax=axes[1, 1], fraction=0.046, pad=0.04)
        cbar.set_label("Percent")

    plt.tight_layout()
    plt.show()

    qc_cols = [
        "run_id",
        "run_key",
        "subject",
        "deployment",
        "samples_requested",
        "samples_valid",
        "expected_answer_points",
        "realized_answer_points",
        "recent_window_points",
    ]
    display(run_df[qc_cols].sort_values("realized_answer_points").head(20))


In [ ]:
# Drill into one run_key (edit RUN_KEY to inspect another run)
if run_df.empty:
    print("run_df is empty")
else:
    RUN_KEY = run_df.sort_values("realized_answer_points", ascending=False).iloc[0]["run_key"]
    print(f"RUN_KEY={RUN_KEY}")
    plot_single_run_window(RUN_KEY, sample_long_df=sample_long_df, rolling_samples=5)
